# Chapter 9 &mdash; A Non-Trivial Conversion, Step by Step

**Concept 4 of the Chapter 9 decomposition:** *A Non-Trivial Conversion, Step by Step*

A looping NFA converted by eliminating states one at a time, with every substitute edge shown.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-Non-Trivial-Conversion/Concept-Non-Trivial-Conversion.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_NFA2RE     import *
from jove.AnimateNFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateNFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateNFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


The whole algorithm on one machine, with nothing skipped.

Take an NFA with a genuine loop, wrap it into a GNFA, and eliminate states one at a
time. At each step:

* the chosen state's **self-loop** is starred;
* every in&ndash;out pair gets a **bypass** edge;
* labels already present are **unioned**.

`del_gnfa_states` prints which state it eliminates and returns the final GNFA, a list
of drawings (one per step), and the resulting RE string. Reading that trace is the
fastest way to internalise the rule.

## 2. Definitions

### A looping NFA

In [ ]:
N = md2mc('''NFA
I : 0 -> I
I : 1 -> A
A : 0 -> A
A : 1 -> F
F : 0 -> I
F : 1 -> F
''')
print("states :", sorted(N["Q"]))

### Convert, keeping every intermediate drawing

In [ ]:
def convert(N, dellist=None):
    g = mk_gnfa(N)
    out = del_gnfa_states(g) if dellist is None else del_gnfa_states(g, DelList=dellist)
    Gfinal, drawings, restr = out
    return Gfinal, drawings, restr

<!-- nav-strip -->

---

&larr;&nbsp;[Ch9&nbsp;3.&nbsp;Exponential Blow-Up in NFA-to-RE Conversion](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-Exponential-Blow-Up-NFA2RE/Concept-Exponential-Blow-Up-NFA2RE.ipynb) &nbsp;&middot;&nbsp; [**Chapter 9** index](https://github.com/ganeshutah/Jove/blob/master/Chapter9/README.md) &nbsp;&middot;&nbsp; [Ch9&nbsp;5.&nbsp;Checking the Conversion by Round-Tripping to Minimal DFA](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-Round-Trip-Check/Concept-Round-Trip-Check.ipynb)&nbsp;&rarr;

---

## 3. Tests

Run it and watch the elimination order.

In [ ]:
Gf, drawings, restr = convert(N)
print("\nfinal GNFA states :", sorted(Gf["Q"]))
print("drawings produced :", len(drawings))
print("\nRE :", restr)
assert sorted(Gf["Q"]) == ['Real_F', 'Real_I']

The RE round-trips to an isomorphic minimal DFA &mdash; the conversion is correct.

In [ ]:
D_orig = min_dfa(nfa2dfa(N))
D_re   = min_dfa(nfa2dfa(re2nfa(restr)))
print("original minimal : %d states" % len(D_orig["Q"]))
print("RE minimal       : %d states" % len(D_re["Q"]))
print("iso_dfa          :", iso_dfa(D_orig, D_re))
assert iso_dfa(D_orig, D_re)

String by string, the RE and the NFA agree.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(11) for p in product('01', repeat=k)]
bad = [s for s in strs if accepts_nfa(N, s) != accepts_dfa(D_re, s)]
print("mismatches over %d strings :" % len(strs), bad)
assert not bad

Forcing a different order gives a different RE for the same language.

In [ ]:
for order in [['I', 'A', 'F'], ['F', 'A', 'I'], ['A', 'I', 'F']]:
    _, _, r = convert(N, order)
    D = min_dfa(nfa2dfa(re2nfa(r)))
    print("order %-18s len %3d  iso: %s" % (order, len(r), iso_dfa(D, D_orig)))
    assert iso_dfa(D, D_orig)

`choose_state_to_del` is the heuristic Jove uses when you do not name an order.

In [ ]:
g = mk_gnfa(N)
left = sorted(g["Q"] - {"Real_I", "Real_F"})
print("states available :", left)
print("heuristic picks  :", choose_state_to_del(g, left))

## 4. Animation

The machine before conversion.

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(N, FuseEdges=True)

## 5. Exercises


1. Do the conversion by hand for the order `['A','F','I']` and compare with Jove's.
2. Which elimination produced the longest label? Why that one?
3. Convert a machine with **two** final states. Where does the union appear?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter9/Concept-Non-Trivial-Conversion')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')